## In this file I will demonstrate that the fix I have used for resolving the non-convergence issue works and eliminates the need for discrete rewards.


### Initialize the network and introduce the outage scenario

In [1]:
from DSS_Initialize import *
from Check_state_action_reward import *
import numpy as np

In [2]:
# Initialize circuit and graph
DSSCktobj, G_init, conv_flag = initialize()
G = G_init.copy()

In [3]:
custom_outage = [('34', '13'), ('23', '21'), ('24', '23'), ('9', '8'), ('8', '7')]

In [4]:
for (u, v) in custom_outage:
    branch_names = G_init.edges[(u, v)]["label"]
    for br_name in branch_names:
        DSSCktobj.dss.Text.Command(f"Open {br_name} term=1")  # Open the line in OpenDSS

In [5]:
G.remove_edges_from(custom_outage)

In [6]:
obs = get_state(DSSCktobj, G, custom_outage)

In [7]:
obs

{'EnergySupp': array([1.00765891]),
 'NodeFeat(BusVoltage)': array([[0.99999699, 0.99999993, 0.99999827],
        [1.01874581, 1.01874987, 1.01874758],
        [1.01874565, 1.01874978, 1.01874744],
        [1.01257532, 1.01953824, 1.0131873 ],
        [0.        , 1.01931243, 0.        ],
        [0.        , 0.        , 1.01154257],
        [1.00799213, 1.0202159 , 1.00995641],
        [0.        , 0.        , 1.0110222 ],
        [0.        , 0.        , 1.0102503 ],
        [0.        , 0.        , 1.00958538],
        [1.00509251, 1.02059689, 1.00782089],
        [0.        , 1.02030685, 0.        ],
        [1.00407167, 0.        , 0.        ],
        [1.00146179, 1.02096   , 1.00462156],
        [1.00407115, 0.        , 0.        ],
        [1.00325655, 0.        , 0.        ],
        [0.        , 0.        , 1.00363445],
        [0.99629247, 1.01818412, 0.99628077],
        [1.00310647, 0.        , 0.        ],
        [1.0029275 , 0.        , 0.        ],
        [0.        ,

### Implement the Hodge L0 solution - previous

In [8]:
HodgeL0_action = np.array([1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.,
       1., 1., 1., 1., 0., 1., 1., 0., 0., 1., 1., 0., 1., 0., 1., 1., 0.,
       1., 0., 0., 1., 0., 0., 0.])

In [9]:
DSSCktobj_sc, G_sc = take_action(HodgeL0_action, custom_outage)

In [10]:
new_obs = get_state(DSSCktobj_sc, G_sc, custom_outage)

In [11]:
new_obs

{'EnergySupp': array([0.05722638]),
 'NodeFeat(BusVoltage)': array([[9.99999479e-01, 9.99999827e-01, 9.99999125e-01],
        [1.00624929e+00, 1.00624976e+00, 1.00624881e+00],
        [1.00624928e+00, 1.00624976e+00, 1.00624879e+00],
        [1.00600307e+00, 1.00628994e+00, 1.00499448e+00],
        [0.00000000e+00, 1.00606116e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 1.00334982e+00],
        [1.00580787e+00, 1.00640162e+00, 1.00497611e+00],
        [0.00000000e+00, 0.00000000e+00, 1.00282520e+00],
        [0.00000000e+00, 0.00000000e+00, 1.00206455e+00],
        [0.00000000e+00, 0.00000000e+00, 1.00140502e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
        [0.0

In [12]:
get_reward(new_obs)

array([0.])

### Fix the non-convergence issue

In [13]:
converged = DSSCktobj_sc.dss.Solution.Converged()

In [14]:
converged

False

In [14]:
if not converged:
    components = list(nx.connected_components(G_sc))
    invalid_components = []
    for bus in node_list:
        DSSCktobj_sc.dss.Circuit.SetActiveBus(bus)
        basekv = DSSCktobj_sc.dss.Bus.kVBase()
        if basekv == 0 or math.isnan(basekv):
           print(f"Bus {bus} has invalid basekv: {basekv}",flush =True)
           for component in components:
               if bus in component and substatn_id not in component:
                  print(f"Invalid Components")
                  invalid_components.append(component)

### No invalid basekv buses!

### Next step deactivate the network components without any sources

In [16]:
Components = list(nx.connected_components(G_sc)) #components formed due to outage
Virtual_Buses = [] # for each component not connected to sourcebus...we will assign a slack
if len(Components)>1 : #Only if there exists a network component unconnected to sourcebus virtual slack is assigned
   for C in Components: #for each component
         if substatn_id not in C: # for the component unconnected to sourcebus
             for gen_bus, gen_info in Gen_Info.items():
                 if gen_bus in C and gen_info['Blackstart']==1: #if generator is present and has gridforming capability
                    Virtual_Buses.append(gen_bus)

In [17]:
Virtual_Buses

['39', '46', '108', '96', '71', '75', '79']

In [25]:
unsupp_buses = []
for C in Components: #for each component
    flag = 1
    if substatn_id not in C:
       flag = 0
       for vb in Virtual_Buses:
           if vb in C:
               flag = 1
               break
    if flag == 0:
       for b in C:
           unsupp_buses.append(b)

In [28]:
for b in unsupp_buses:
    flag = DSSCktobj_sc.dss.Circuit.SetActiveBus(b)
    if flag>0:
       comp_elements = DSSCktobj_sc.dss.Bus.AllPCEatBus()
       for celem in comp_elements:
           DSSCktobj_sc.dss.Text.Command(f"Edit {celem} enabled=False")

In [31]:
DSSCktobj_sc.dss.Text.Command("Set controlmode = OFF")

In [32]:
DSSCktobj_sc.dss.Solution.Solve()

In [33]:
DSSCktobj_sc.dss.Solution.Converged()

False